# C11-neural-training — Practice p10 — Solution


**Type:** constrained coding · **Difficulty:** core · **Concepts:** dropout


One module is switched between modes. Resetting the seed before replay restores
the same random-number sequence; evaluation uses the identity map.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

def dropout_train_eval(x, p, seed=20260804):
    values = x.detach().clone().to(dtype=torch.float64, device="cpu")
    drop = nn.Dropout(p=p)
    drop.train()
    torch.manual_seed(seed)
    first, second = drop(values), drop(values)
    torch.manual_seed(seed)
    repeat_first, repeat_second = drop(values), drop(values)
    drop.eval()
    eval_first, eval_second = drop(values), drop(values)
    return {k: v.detach().clone() for k, v in {
        "train_first": first, "train_second": second, "repeat_first": repeat_first,
        "repeat_second": repeat_second, "eval_first": eval_first, "eval_second": eval_second}.items()}

x_p10 = torch.linspace(-3.0, 3.0, 101)
x_p10[torch.abs(x_p10) < 1e-9] = 0.125
result_p10 = dropout_train_eval(x_p10, p=0.25)


### Answer check


In [ ]:
assert torch.all(torch.abs(x_p10) >= 1e-9)
expected_kept_p10 = x_p10 / 0.75
for key_p10 in ("train_first", "train_second"):
    actual_p10 = result_p10[key_p10]
    dropped_p10 = actual_p10 == 0.0
    kept_p10 = torch.isclose(actual_p10, expected_kept_p10, atol=1e-12, rtol=1e-12)
    assert torch.all(dropped_p10 | kept_p10)
assert torch.equal(result_p10["train_first"], result_p10["repeat_first"])
assert torch.equal(result_p10["train_second"], result_p10["repeat_second"])
assert not torch.equal(result_p10["train_first"], result_p10["train_second"])
assert torch.allclose(result_p10["eval_first"], x_p10, atol=1e-12, rtol=1e-12)
assert torch.allclose(result_p10["eval_second"], x_p10, atol=1e-12, rtol=1e-12)
